In [ ]:
import sys
from pathlib import Path

# add root directory to sys.path to enable importing utils.pipeline
PROJECT_ROOT_DIR = str(Path.cwd().parent)
if PROJECT_ROOT_DIR not in sys.path:
    sys.path.append(PROJECT_ROOT_DIR)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    classification_report,
    confusion_matrix,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split

from utils import YN_MAP, pipeline

In [ ]:
%matplotlib inline
sns.set_style('whitegrid')

In [ ]:
# load dataset
df = pd.read_csv('../data/TelcoCustomerChurn.csv')

In [ ]:

X = df.drop(columns=['Churn'])
y = df['Churn'].map(YN_MAP)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
print("\nTraining Set:")
print("X_train", X_train.shape)
print("y_train", y_train.value_counts())
print("y_train %", y_train.value_counts(normalize=True).mul(100).round(2))

print("\nTest Set:")
print("X_test", X_test.shape)
print("y_test", y_test.value_counts())
print("y_test %", y_test.value_counts(normalize=True).mul(100).round(2))


In [ ]:
pipeline.fit(X_train, y_train)
train_acc1 = pipeline.score(X_train, y_train)
test_acc1 = pipeline.score(X_test, y_test)
print(f"Baseline — Train acc: {train_acc1:.3f} | Test acc: {test_acc1:.3f}")

y_pred1 = pipeline.predict(X_test)
y_proba1 = pipeline.predict_proba(X_test)[:, 1]
print(classification_report(y_test, y_pred1))
# print(classification_report(y_test, y_pred1, target_names=['No Churn', 'Churn']))
print(f"ROC-AUC: {roc_auc_score(y_test, y_proba1):.3f}")

# confusion matrix - nos
cm = confusion_matrix(y_test, y_pred1)
# disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No Churn', 'Churn'])
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap='Blues', values_format='d')
plt.title('Confusion Matrix — Nos')
plt.show()
# confusion matrix - pct
cm_pct = confusion_matrix(y_test, y_pred1, normalize='true')
# disp = ConfusionMatrixDisplay(confusion_matrix=cm_pct, display_labels=['No Churn', 'Churn'])
disp = ConfusionMatrixDisplay(confusion_matrix=cm_pct)
disp.plot(cmap='Blues', values_format='.2%')
plt.title('Confusion Matrix — %')
plt.show()

In [ ]:
from datetime import datetime, timedelta, timezone

import joblib

timestamp = datetime.now(tz=timezone(timedelta(hours=5, minutes=30))).strftime("%Y%m%d%H%M%S")
targets = [
    '../model/telco_churn_pipeline_latest.joblib',
    f'../model/telco_churn_pipeline_{timestamp}.joblib'
]
for target_file_path in targets:
    joblib.dump(pipeline, target_file_path)

In [ ]:
pipeline = joblib.load('../model/telco_churn_pipeline_latest.joblib')

new_customer = pd.DataFrame([{
    'customerID': '9999-XYZ', 'gender': 'Female', 'SeniorCitizen': 0, 'Partner': 'Yes',
    'Dependents': 'No', 'tenure': 5, 'PhoneService': 'Yes', 'MultipleLines': 'No',
    'InternetService': 'Fiber optic', 'OnlineSecurity': 'No', 'OnlineBackup': 'No',
    'DeviceProtection': 'No', 'TechSupport': 'No', 'StreamingTV': 'No', 'StreamingMovies': 'No',
    'Contract': 'Month-to-month', 'PaperlessBilling': 'Yes', 'PaymentMethod': 'Electronic check',
    'MonthlyCharges': 85.0, 'TotalCharges': '425.0'
}])

pred = pipeline.predict(new_customer)
proba = pipeline.predict_proba(new_customer)[:, 1]
print(f"Churn: {pred[0]}, probability: {proba[0]:.3f}")